# Liquid-Cooling Rack Leak Detection Demo

This notebook simulates a liquid-cooling rack with a controllable inlet valve, two pressure sensors (`p1`, `p2`), and `N` active tray branches in parallel.

## Assumptions

- Quasi-steady, incompressible flow
- Square-law pressure losses of the form `delta_p = R * q^2`
- Identical tray branches unless a tray-branch leak is injected
- One small return-line resistance is included so that `p2` changes with flow and remains informative
- Leak flow is modeled as an extra hydraulic branch discharging to return
- The valve is a variable resistance with a numerically safe lower bound on opening

## What the notebook covers

1. Core hydraulic model
2. Scenario runner
3. Residual-based leak detection
4. Plotly visualizations
5. Interactive controls with `ipywidgets`
6. Demo cases including valve shutdown response

The intent is a clear proof of concept rather than a high-fidelity CFD or transient fluid-network model.


In [3]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from ipywidgets import Dropdown, FloatSlider, HBox, IntSlider, Output, VBox, interactive_output
from plotly.subplots import make_subplots
from scipy.optimize import least_squares

pd.set_option("display.float_format", lambda value: f"{value:0.4f}")


def render_plotly_figure(fig: go.Figure) -> None:
    try:
        display(fig)
    except Exception:
        html = fig.to_html(include_plotlyjs="cdn", full_html=False)
        display(HTML(html))


ModuleNotFoundError: No module named 'ipywidgets'

In [ ]:
EPS = 1e-9


def safe_opening(opening: float, minimum: float = 1e-3) -> float:
    return float(np.clip(opening, minimum, 1.0))


def hydraulic_flow(delta_p: float, resistance: float) -> float:
    resistance = max(float(resistance), EPS)
    return np.sqrt(max(float(delta_p), 0.0) / resistance)


@dataclass(frozen=True)
class RackParameters:
    supply_pressure: float = 2.2
    return_pressure: float = 0.0
    tray_resistance: float = 0.085
    return_resistance: float = 0.020
    valve_r_min: float = 0.012
    valve_k: float = 0.055
    opening_epsilon: float = 0.03


@dataclass(frozen=True)
class Scenario:
    name: str
    valve_opening: float
    active_trays: int
    leak_position: str = "none"
    leak_resistance: float = np.inf


@dataclass
class SimulationResult:
    scenario_name: str
    valve_opening: float
    active_trays: int
    leak_position: str
    leak_resistance: float
    p1: float
    p2: float
    dp_rack: float
    total_inlet_flow: float
    tray_flows: List[float]
    leak_flow: float
    effective_rack_resistance: float
    valve_resistance: float
    return_flow: float
    solved: bool
    solver_cost: float
    expected_dp_rack: float = np.nan
    expected_total_flow: float = np.nan
    dp_residual: float = np.nan
    flow_residual: float = np.nan
    leak_score: float = np.nan
    detection: str = "unknown"
    shutdown_valve_opening: float = np.nan
    shutdown_total_flow: float = np.nan
    shutdown_leak_flow: float = np.nan
    shutdown_dp_rack: float = np.nan

    def to_record(self) -> Dict[str, float]:
        record = asdict(self)
        record["tray_flows"] = np.asarray(self.tray_flows, dtype=float)
        return record


In [ ]:
class HydraulicRackModel:
    def __init__(self, params: RackParameters):
        self.params = params

    def valve_resistance(self, opening: float) -> float:
        opening = safe_opening(opening, self.params.opening_epsilon)
        return self.params.valve_r_min + self.params.valve_k / (opening ** 2)

    def _residuals(
        self,
        x: np.ndarray,
        opening: float,
        active_trays: int,
        leak_position: str,
        leak_resistance: float,
    ) -> np.ndarray:
        params = self.params
        p_supply = params.supply_pressure
        p_return = params.return_pressure
        r_valve = self.valve_resistance(opening)
        leak_on = np.isfinite(leak_resistance) and leak_position != "none"

        if leak_position in {"middle", "tray_branch"}:
            p1, pmid, p2 = x
        else:
            p1, p2 = x
            pmid = np.nan

        q_valve = hydraulic_flow(p_supply - p1, r_valve)
        q_return = hydraulic_flow(p2 - p_return, params.return_resistance)

        if leak_position in {"none", "inlet", "outlet"}:
            q_tray_each = hydraulic_flow(p1 - p2, params.tray_resistance)
            q_trays = active_trays * q_tray_each

            q_leak = 0.0
            if leak_on and leak_position == "inlet":
                q_leak = hydraulic_flow(p1 - p_return, leak_resistance)
            elif leak_on and leak_position == "outlet":
                q_leak = hydraulic_flow(p2 - p_return, leak_resistance)

            return np.array(
                [
                    q_valve - q_trays - (q_leak if leak_position == "inlet" else 0.0),
                    q_trays - q_return - (q_leak if leak_position == "outlet" else 0.0),
                ]
            )

        if leak_position == "middle":
            r_half = 0.5 * params.tray_resistance
            q_in_each = hydraulic_flow(p1 - pmid, r_half)
            q_out_each = hydraulic_flow(pmid - p2, r_half)
            q_in = active_trays * q_in_each
            q_out = active_trays * q_out_each
            q_leak = hydraulic_flow(pmid - p_return, leak_resistance) if leak_on else 0.0
            return np.array(
                [
                    q_valve - q_in,
                    q_in - q_out - q_leak,
                    q_out - q_return,
                ]
            )

        if leak_position == "tray_branch":
            r_half = 0.5 * params.tray_resistance
            intact_count = max(active_trays - 1, 0)
            q_intact_each = hydraulic_flow(p1 - p2, params.tray_resistance)
            q_intact = intact_count * q_intact_each
            q_leaky_in = hydraulic_flow(p1 - pmid, r_half)
            q_leaky_out = hydraulic_flow(pmid - p2, r_half)
            q_leak = hydraulic_flow(pmid - p_return, leak_resistance) if leak_on else 0.0
            return np.array(
                [
                    q_valve - q_intact - q_leaky_in,
                    q_leaky_in - q_leaky_out - q_leak,
                    q_intact + q_leaky_out - q_return,
                ]
            )

        raise ValueError(f"Unsupported leak position: {leak_position}")

    def solve(
        self,
        opening: float,
        active_trays: int,
        leak_position: str = "none",
        leak_resistance: float = np.inf,
        scenario_name: str = "scenario",
    ) -> SimulationResult:
        if active_trays < 1:
            raise ValueError("active_trays must be at least 1")

        params = self.params
        p_supply = params.supply_pressure
        p_return = params.return_pressure
        leak_position = leak_position.lower()

        if leak_position in {"middle", "tray_branch"}:
            x0 = np.array([0.78 * p_supply, 0.52 * p_supply, 0.18 * p_supply], dtype=float)
            lower = np.array([p_return, p_return, p_return], dtype=float)
            upper = np.array([p_supply, p_supply, p_supply], dtype=float)
        else:
            x0 = np.array([0.72 * p_supply, 0.20 * p_supply], dtype=float)
            lower = np.array([p_return, p_return], dtype=float)
            upper = np.array([p_supply, p_supply], dtype=float)

        solution = least_squares(
            self._residuals,
            x0=x0,
            bounds=(lower, upper),
            args=(opening, active_trays, leak_position, leak_resistance),
            xtol=1e-12,
            ftol=1e-12,
            gtol=1e-12,
            max_nfev=500,
        )

        if not solution.success:
            raise RuntimeError(f"Hydraulic solve failed: {solution.message}")

        if leak_position in {"middle", "tray_branch"}:
            p1, pmid, p2 = solution.x
        else:
            p1, p2 = solution.x
            pmid = np.nan

        q_valve = hydraulic_flow(p_supply - p1, self.valve_resistance(opening))
        q_return = hydraulic_flow(p2 - p_return, params.return_resistance)

        if leak_position in {"none", "inlet", "outlet"}:
            tray_flows = [hydraulic_flow(p1 - p2, params.tray_resistance)] * active_trays
            leak_flow = 0.0
            if np.isfinite(leak_resistance) and leak_position == "inlet":
                leak_flow = hydraulic_flow(p1 - p_return, leak_resistance)
            elif np.isfinite(leak_resistance) and leak_position == "outlet":
                leak_flow = hydraulic_flow(p2 - p_return, leak_resistance)

        elif leak_position == "middle":
            r_half = 0.5 * params.tray_resistance
            tray_flows = [hydraulic_flow(p1 - pmid, r_half)] * active_trays
            leak_flow = hydraulic_flow(pmid - p_return, leak_resistance) if np.isfinite(leak_resistance) else 0.0

        elif leak_position == "tray_branch":
            r_half = 0.5 * params.tray_resistance
            intact_tray_flow = hydraulic_flow(p1 - p2, params.tray_resistance)
            leaky_tray_flow = hydraulic_flow(p1 - pmid, r_half)
            tray_flows = [intact_tray_flow] * max(active_trays - 1, 0) + [leaky_tray_flow]
            leak_flow = hydraulic_flow(pmid - p_return, leak_resistance) if np.isfinite(leak_resistance) else 0.0

        else:
            raise ValueError(f"Unsupported leak position: {leak_position}")

        dp_rack = p1 - p2
        q_for_effective_r = max(q_valve, EPS)
        effective_rack_resistance = dp_rack / (q_for_effective_r ** 2)

        return SimulationResult(
            scenario_name=scenario_name,
            valve_opening=float(opening),
            active_trays=int(active_trays),
            leak_position=leak_position,
            leak_resistance=float(leak_resistance),
            p1=float(p1),
            p2=float(p2),
            dp_rack=float(dp_rack),
            total_inlet_flow=float(q_valve),
            tray_flows=[float(value) for value in tray_flows],
            leak_flow=float(leak_flow),
            effective_rack_resistance=float(effective_rack_resistance),
            valve_resistance=float(self.valve_resistance(opening)),
            return_flow=float(q_return),
            solved=bool(solution.success),
            solver_cost=float(solution.cost),
        )


In [ ]:
@dataclass(frozen=True)
class DetectorConfig:
    dp_threshold: float = 0.080
    flow_threshold: float = 0.100
    score_threshold: float = 1.0
    tray_change_margin: float = 0.15
    min_opening_for_detection: float = 0.10


class ResidualLeakDetector:
    def __init__(self, model: HydraulicRackModel, config: DetectorConfig, tray_candidates: Optional[List[int]] = None):
        self.model = model
        self.config = config
        self.tray_candidates = tray_candidates or list(range(1, 17))

    def baseline_result(self, opening: float, active_trays: int) -> SimulationResult:
        return self.model.solve(
            opening=opening,
            active_trays=active_trays,
            leak_position="none",
            leak_resistance=np.inf,
            scenario_name=f"baseline_N{active_trays}_u{opening:0.2f}",
        )

    def evaluate(self, result: SimulationResult) -> SimulationResult:
        baseline = self.baseline_result(result.valve_opening, result.active_trays)
        dp_residual = result.dp_rack - baseline.dp_rack
        flow_residual = result.total_inlet_flow - baseline.total_inlet_flow

        dp_norm = abs(dp_residual) / max(self.config.dp_threshold, EPS)
        flow_norm = abs(flow_residual) / max(self.config.flow_threshold, EPS)
        leak_score = max(dp_norm, flow_norm)

        detection = "normal"
        if result.valve_opening < self.config.min_opening_for_detection:
            detection = "low_signal"
        elif leak_score >= self.config.score_threshold:
            best_candidate = None
            best_norm = np.inf
            for candidate in self.tray_candidates:
                candidate_baseline = self.baseline_result(result.valve_opening, candidate)
                candidate_dp_norm = abs(result.dp_rack - candidate_baseline.dp_rack) / max(self.config.dp_threshold, EPS)
                candidate_flow_norm = abs(result.total_inlet_flow - candidate_baseline.total_inlet_flow) / max(self.config.flow_threshold, EPS)
                candidate_score = max(candidate_dp_norm, candidate_flow_norm)
                if candidate_score < best_norm:
                    best_norm = candidate_score
                    best_candidate = candidate

            if (
                best_candidate is not None
                and best_candidate != result.active_trays
                and best_norm < self.config.score_threshold
                and best_norm < leak_score * (1.0 - self.config.tray_change_margin)
            ):
                detection = f"tray_change_suspected_to_{best_candidate}"
            else:
                detection = "leak_detected"

        result.expected_dp_rack = baseline.dp_rack
        result.expected_total_flow = baseline.total_inlet_flow
        result.dp_residual = dp_residual
        result.flow_residual = flow_residual
        result.leak_score = leak_score
        result.detection = detection
        return result


def apply_shutdown(
    model: HydraulicRackModel,
    result: SimulationResult,
    closed_opening: float = 0.03,
) -> SimulationResult:
    if result.detection != "leak_detected":
        result.shutdown_valve_opening = result.valve_opening
        result.shutdown_total_flow = result.total_inlet_flow
        result.shutdown_leak_flow = result.leak_flow
        result.shutdown_dp_rack = result.dp_rack
        return result

    shutdown = model.solve(
        opening=closed_opening,
        active_trays=result.active_trays,
        leak_position=result.leak_position,
        leak_resistance=result.leak_resistance,
        scenario_name=f"{result.scenario_name}_shutdown",
    )
    result.shutdown_valve_opening = closed_opening
    result.shutdown_total_flow = shutdown.total_inlet_flow
    result.shutdown_leak_flow = shutdown.leak_flow
    result.shutdown_dp_rack = shutdown.dp_rack
    return result


def simulate_scenario(
    model: HydraulicRackModel,
    detector: ResidualLeakDetector,
    scenario: Scenario,
) -> SimulationResult:
    result = model.solve(
        opening=scenario.valve_opening,
        active_trays=scenario.active_trays,
        leak_position=scenario.leak_position,
        leak_resistance=scenario.leak_resistance,
        scenario_name=scenario.name,
    )
    result = detector.evaluate(result)
    result = apply_shutdown(model, result)
    return result


def results_to_frame(results: List[SimulationResult]) -> pd.DataFrame:
    rows = []
    max_trays = max(len(result.tray_flows) for result in results)
    for result in results:
        row = result.to_record()
        tray_flows = list(result.tray_flows) + [np.nan] * (max_trays - len(result.tray_flows))
        row["mean_tray_flow"] = float(np.mean(result.tray_flows))
        row["max_tray_flow"] = float(np.max(result.tray_flows))
        row["min_tray_flow"] = float(np.min(result.tray_flows))
        for idx, value in enumerate(tray_flows, start=1):
            row[f"tray_{idx}_flow"] = value
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
params = RackParameters(
    supply_pressure=2.2,
    return_pressure=0.0,
    tray_resistance=0.085,
    return_resistance=0.020,
    valve_r_min=0.012,
    valve_k=0.055,
    opening_epsilon=0.03,
)

model = HydraulicRackModel(params)
detector = ResidualLeakDetector(
    model,
    DetectorConfig(
        dp_threshold=0.070,
        flow_threshold=0.085,
        score_threshold=1.0,
        tray_change_margin=0.10,
        min_opening_for_detection=0.08,
    ),
    tray_candidates=list(range(1, 13)),
)

baseline_example = detector.baseline_result(opening=0.65, active_trays=8)
pd.DataFrame([baseline_example.to_record()]).drop(columns=["tray_flows"])


## Scenario Runner

We will create a mix of nominal, leak, and tray-count-change scenarios. The detector compares each observed operating point against the expected no-leak baseline for the same valve opening and nominal tray count.


In [ ]:
scenarios = [
    Scenario(name="Normal / 8 trays / 65%", valve_opening=0.65, active_trays=8, leak_position="none", leak_resistance=np.inf),
    Scenario(name="Normal / 5 trays / 65%", valve_opening=0.65, active_trays=5, leak_position="none", leak_resistance=np.inf),
    Scenario(name="Inlet leak / mild", valve_opening=0.65, active_trays=8, leak_position="inlet", leak_resistance=0.38),
    Scenario(name="Middle leak / moderate", valve_opening=0.65, active_trays=8, leak_position="middle", leak_resistance=0.22),
    Scenario(name="Outlet leak / moderate", valve_opening=0.65, active_trays=8, leak_position="outlet", leak_resistance=0.18),
    Scenario(name="Tray branch leak", valve_opening=0.65, active_trays=8, leak_position="tray_branch", leak_resistance=0.11),
    Scenario(name="Open valve / no leak", valve_opening=0.85, active_trays=8, leak_position="none", leak_resistance=np.inf),
    Scenario(name="Open valve / outlet leak", valve_opening=0.85, active_trays=8, leak_position="outlet", leak_resistance=0.15),
]

results = [simulate_scenario(model, detector, scenario) for scenario in scenarios]
results_df = results_to_frame(results)

summary_columns = [
    "scenario_name",
    "valve_opening",
    "active_trays",
    "leak_position",
    "p1",
    "p2",
    "dp_rack",
    "total_inlet_flow",
    "leak_flow",
    "effective_rack_resistance",
    "dp_residual",
    "flow_residual",
    "leak_score",
    "detection",
]

results_df[summary_columns]


In [ ]:
def make_overview_figure(results_frame: pd.DataFrame) -> go.Figure:
    labels = results_frame["scenario_name"]
    fig = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=[
            "Pressures p1 and p2",
            "Rack differential pressure",
            "Total inlet flow",
            "Leak flow",
            "Branch flows",
            "Residual / leak score",
        ],
        vertical_spacing=0.12,
    )

    fig.add_trace(go.Bar(x=labels, y=results_frame["p1"], name="p1"), row=1, col=1)
    fig.add_trace(go.Bar(x=labels, y=results_frame["p2"], name="p2"), row=1, col=1)

    fig.add_trace(go.Bar(x=labels, y=results_frame["dp_rack"], name="dp_rack", marker_color="#2ca02c"), row=1, col=2)
    fig.add_trace(go.Bar(x=labels, y=results_frame["total_inlet_flow"], name="q_in", marker_color="#1f77b4"), row=2, col=1)
    fig.add_trace(go.Bar(x=labels, y=results_frame["leak_flow"], name="q_leak", marker_color="#d62728"), row=2, col=2)

    tray_columns = [column for column in results_frame.columns if column.startswith("tray_") and column.endswith("_flow")]
    for column in tray_columns[: min(8, len(tray_columns))]:
        fig.add_trace(
            go.Scatter(
                x=labels,
                y=results_frame[column],
                mode="lines+markers",
                name=column.replace("_flow", ""),
            ),
            row=3,
            col=1,
        )

    fig.add_trace(
        go.Bar(x=labels, y=results_frame["leak_score"], name="leak_score", marker_color="#9467bd"),
        row=3,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=labels,
            y=np.ones(len(results_frame)) * detector.config.score_threshold,
            mode="lines",
            name="score_threshold",
            line=dict(color="black", dash="dash"),
        ),
        row=3,
        col=2,
    )

    fig.update_layout(
        height=980,
        width=1250,
        barmode="group",
        template="plotly_white",
        title="Leak Detection Scenario Overview",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    )
    fig.update_xaxes(tickangle=25)
    return fig


overview_figure = make_overview_figure(results_df)
render_plotly_figure(overview_figure)


In [ ]:
def make_shutdown_figure(results_frame: pd.DataFrame) -> go.Figure:
    leak_cases = results_frame[results_frame["detection"] == "leak_detected"].copy()
    if leak_cases.empty:
        raise ValueError("No leak_detected scenarios available for shutdown visualization.")

    labels = leak_cases["scenario_name"]
    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=["Valve opening response", "Total flow before/after", "Leak flow before/after"],
    )

    fig.add_trace(
        go.Bar(x=labels, y=leak_cases["valve_opening"], name="before_opening"),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(x=labels, y=leak_cases["shutdown_valve_opening"], name="after_opening"),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Bar(x=labels, y=leak_cases["total_inlet_flow"], name="before_flow"),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Bar(x=labels, y=leak_cases["shutdown_total_flow"], name="after_flow"),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Bar(x=labels, y=leak_cases["leak_flow"], name="before_leak"),
        row=1,
        col=3,
    )
    fig.add_trace(
        go.Bar(x=labels, y=leak_cases["shutdown_leak_flow"], name="after_leak"),
        row=1,
        col=3,
    )

    fig.update_layout(
        height=420,
        width=1250,
        template="plotly_white",
        barmode="group",
        title="Valve Closing Response After Leak Detection",
    )
    fig.update_xaxes(tickangle=25)
    return fig


shutdown_figure = make_shutdown_figure(results_df)
render_plotly_figure(shutdown_figure)


## Interactive Controls

Use the controls below to vary valve opening, active tray count, leak position, and leak resistance. The detector runs immediately and the shutdown state is recomputed if a leak is classified.


In [ ]:
def make_state_table(result: SimulationResult) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"metric": "p1", "value": result.p1},
            {"metric": "p2", "value": result.p2},
            {"metric": "dp_rack", "value": result.dp_rack},
            {"metric": "total_inlet_flow", "value": result.total_inlet_flow},
            {"metric": "leak_flow", "value": result.leak_flow},
            {"metric": "effective_rack_resistance", "value": result.effective_rack_resistance},
            {"metric": "expected_dp_rack", "value": result.expected_dp_rack},
            {"metric": "expected_total_flow", "value": result.expected_total_flow},
            {"metric": "dp_residual", "value": result.dp_residual},
            {"metric": "flow_residual", "value": result.flow_residual},
            {"metric": "leak_score", "value": result.leak_score},
        ]
    )


def plot_single_scenario(result: SimulationResult) -> go.Figure:
    tray_labels = [f"tray_{idx + 1}" for idx in range(len(result.tray_flows))]
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=["Pressures", "Flows", "Branch flows", "Detector residuals"],
        vertical_spacing=0.14,
    )

    fig.add_trace(
        go.Bar(x=["p1", "p2"], y=[result.p1, result.p2], name="pressure", marker_color=["#1f77b4", "#ff7f0e"]),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            x=["inlet", "return", "leak"],
            y=[result.total_inlet_flow, result.return_flow, result.leak_flow],
            name="flow summary",
            marker_color=["#2ca02c", "#17becf", "#d62728"],
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Bar(x=tray_labels, y=result.tray_flows, name="branch_flow", marker_color="#636efa"),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            x=["dp_residual", "flow_residual", "leak_score"],
            y=[result.dp_residual, result.flow_residual, result.leak_score],
            name="detector",
            marker_color=["#9467bd", "#8c564b", "#e377c2"],
        ),
        row=2,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=["dp_residual", "flow_residual", "leak_score"],
            y=[detector.config.dp_threshold, detector.config.flow_threshold, detector.config.score_threshold],
            mode="markers",
            name="threshold reference",
            marker=dict(color="black", size=10, symbol="x"),
        ),
        row=2,
        col=2,
    )

    fig.update_layout(
        height=720,
        width=1100,
        template="plotly_white",
        title=f"{result.scenario_name} | classification: {result.detection}",
        showlegend=False,
    )
    return fig


interactive_output_box = Output()


def render_interactive_case(valve_opening: float, active_trays: int, leak_position: str, leak_resistance: float) -> None:
    scenario = Scenario(
        name="Interactive case",
        valve_opening=valve_opening,
        active_trays=active_trays,
        leak_position=leak_position,
        leak_resistance=(np.inf if leak_position == "none" else leak_resistance),
    )
    result = simulate_scenario(model, detector, scenario)
    table = make_state_table(result)
    figure = plot_single_scenario(result)

    with interactive_output_box:
        interactive_output_box.clear_output(wait=True)
        display(table)
        render_plotly_figure(figure)
        if result.detection == "leak_detected":
            shutdown_df = pd.DataFrame(
                [
                    {"state": "before_shutdown", "total_flow": result.total_inlet_flow, "leak_flow": result.leak_flow, "dp_rack": result.dp_rack},
                    {"state": "after_shutdown", "total_flow": result.shutdown_total_flow, "leak_flow": result.shutdown_leak_flow, "dp_rack": result.shutdown_dp_rack},
                ]
            )
            display(shutdown_df)


controls = {
    "valve_opening": FloatSlider(description="Valve", min=0.03, max=1.0, step=0.01, value=0.65, readout_format=".2f"),
    "active_trays": IntSlider(description="Trays", min=1, max=12, step=1, value=8),
    "leak_position": Dropdown(
        description="Leak pos",
        options=["none", "inlet", "middle", "outlet", "tray_branch"],
        value="none",
    ),
    "leak_resistance": FloatSlider(description="R_leak", min=0.05, max=0.80, step=0.01, value=0.20, readout_format=".2f"),
}

interactive_panel = interactive_output(render_interactive_case, controls)
display(VBox([HBox(list(controls.values())[:2]), HBox(list(controls.values())[2:]), interactive_output_box, interactive_panel]))


## Demo Cases

The table below gives a compact comparison of the requested outputs across all prepared scenarios, including the detector decision and the effect of emergency shutdown.


In [ ]:
demo_columns = [
    "scenario_name",
    "active_trays",
    "valve_opening",
    "leak_position",
    "p1",
    "p2",
    "dp_rack",
    "total_inlet_flow",
    "leak_flow",
    "effective_rack_resistance",
    "detection",
    "shutdown_total_flow",
    "shutdown_leak_flow",
]

results_df[demo_columns].sort_values(by=["scenario_name"]).reset_index(drop=True)


## Engineering Notes

### What is simplified

- The model is quasi-steady, so there is no fluid capacitance, pipe elasticity, transport delay, or transient wave dynamics.
- All tray branches are represented by single square-law resistances.
- Fluid properties are constant and temperature effects are ignored.
- Leak paths discharge directly to return through a single resistance.
- Sensor dynamics, noise, and calibration offsets are not modeled.

### What would change in a more realistic model

- Add pipe volumes and compressibility-like storage terms for transient pressure behavior.
- Split manifolds, hoses, trays, and quick-connects into more detailed subcomponents.
- Include pump curves, valve dynamics, actuator limits, and controller latency.
- Model sensor noise, filtering, and threshold tuning from measured operating data.
- Add uncertainty and parameter variation across trays and hardware states.

### How this could later map to Simscape

- Represent the valve, return line, trays, and leak paths as Simscape Fluids hydraulic resistive elements.
- Add pipe or chamber compliance to capture transient response and leak propagation timing.
- Use pressure sensors at the `p1` and `p2` nodes and implement the residual detector in Simulink.
- Drive shutdown logic with a supervisory state machine that commands the valve toward closed when the residual exceeds threshold.
